# HNSW — Hierarchical Navigable Small-World Construction and Search

The narrative companion to `hnsw.py`, which **owns every number**. The prerequisite (navigable
small-world graphs) gave us a single navigable graph and a beam search that walks it; its honest
catch is that the walk starts at an arbitrary entry and the small-world property is empirical. HNSW
adds one structural idea — a **randomized hierarchy of nested graphs** — that turns the arbitrary
entry into a provably logarithmic descent. We walk the three movements and call the verification
harness; each assertion encodes a claim the topic makes.

In [ ]:
import sys, pathlib
_here = pathlib.Path.cwd()
for _cand in (_here, _here / "notebooks" / "hnsw", pathlib.Path("notebooks/hnsw")):
    if (_cand / "hnsw.py").exists():
        sys.path.insert(0, str(_cand)); break
import numpy as np
import hnsw

## Movement 1 — the level-assignment law (the provable spine)

Each node draws a maximum level $L=\lfloor -\ln(U)\,m_L\rfloor$ with $U\sim\mathrm{Uniform}(0,1)$
and $m_L=1/\ln M$, and is inserted into the graphs at every layer $0,\dots,L$. Then
$P(L\ge\ell)=M^{-\ell}$ **exactly**, so layer occupancy decays geometrically by $1/M$, the top layer
holds $O(1)$ nodes, and the expected maximum level — the entry-descent depth — grows like
$\log_M n$. This is exact probability, the analogue of the prerequisite's Kleinberg theorem.

In [ ]:
hnsw.test_level_tail_law()
hnsw.test_layer_geometric_decay()
hnsw.test_top_layer_is_O1()
hnsw.test_max_level_scales_log()

## Movement 2 — heuristic neighbor selection (the centerpiece, and a heuristic)

What distinguishes HNSW from "link to the $M$ nearest" is Malkov–Yashunin **Algorithm 4**: scanning
candidates by increasing distance to the base, admit one only if no already-kept neighbor is closer to
it than it is to the base. Strict-$M$-nearest clusters links on one side; the heuristic spreads them
and preserves the long-range links that keep the graph a small world. It has **no optimality proof** —
we show only the direction in which it differs from naive selection.

In [ ]:
hnsw.test_heuristic_diversifies()

## Movement 3 — hierarchical construction and search

Construction inserts each point at its level, reusing the prerequisite's beam **restricted to a
per-layer adjacency**. The correctness anchor: forced to a single layer, the fresh `search_layer` is
the prerequisite's `greedy_search` exactly — same indices *and* same distance-computation count — so
the hierarchy is the only new thing, not the search. Search descends greedily (beam 1) through the
sparse upper layers to refine the entry, then runs a width-`ef` beam at layer 0; recall is
non-decreasing in `ef`.

In [ ]:
hnsw.test_search_layer_matches_flat_on_single_layer()
hnsw.test_recall_monotone_in_ef_hnsw()

## The arc-closing head-to-head: HNSW vs flat NSW vs IVF, on one cloud

We build all three indexes on the **same** `(X, queries)` with one shared ground truth and trace
recall-versus-cost frontiers (cost = mean exact distance computations per query). The **robust,
intra-graph headline** is that the hierarchy reaches a given recall at no more cost than the flat NSW
it layers. Against the inverted file we report — but do not universalize — the comparison at a matched
cost budget on this one synthetic low-rank cloud.

In [ ]:
hnsw.test_hnsw_beats_flat_nsw_at_equal_cost()
hnsw.test_both_reach_exact_at_full_cost()
hnsw.test_hnsw_vs_ivf_on_one_cloud()

## Viz constants

The numbers `HNSWLaboratory.tsx` mirrors to the decimal: the toy multi-layer graph and one query's
descent (Panel A), the naive-vs-heuristic neighbor sets (Panel B), and the recall/cost frontiers plus
the $\log_M n$ scaling (Panel C).

In [ ]:
hnsw.viz_constants()

## All claims verified

The level law is exact; the geometric decay, the $\log_M n$ scaling, the single-layer equivalence, the
heuristic's diversification, monotone recall, and the hierarchy's cost advantage are all asserted
above. The honest caveats — empirical end-to-end $O(\log n)$ on real data, the unproven neighbor
heuristic, recall's dependence on $M$/`ef`/$m_L$/entry, and the single-cloud head-to-head — are carried
in the topic's rigorFlag.